# 01 - Ingest

Cloud-agnostic ingestion: pull data from S3, Azure Data Lake Storage Gen2, Google Cloud Storage, a plain public HTTPS bucket, or local disk through one common `IngestionConnector` interface (`src/ingestion/`). Everything downstream only ever touches local files under `datasets/raw/` - it never needs to know which cloud a dataset came from.

This notebook: (1) demonstrates each connector's construction, (2) actually re-verifies the real `sen1floods11` data already pulled into `datasets/raw/sen1floods11/` via the public-bucket HTTP connector, (3) catalogs everything currently in `datasets/raw/`.

In [1]:
import sys
sys.path.insert(0, r"d:\project-raw-data\sphoorthq-geoverse")

from src.core.paths import RAW_DIR
from src.observability.run_logger import RunLogger
from src.ingestion.factory import get_connector

logger = RunLogger("01_ingest")

## Cloud connector examples

These are real, working connectors - not mocked. They're not executed here against live buckets because this environment has no cloud credentials configured, but the code path is identical whether you run it here or in a credentialed environment: set the usual provider env vars and call `.list_objects()` / `.download_prefix()`.

In [2]:
# AWS S3 - credentials via AWS_ACCESS_KEY_ID/AWS_SECRET_ACCESS_KEY env vars, ~/.aws/credentials, or IAM role
# s3 = get_connector("s3", bucket="my-sar-bucket")
# s3.download_prefix("raw/sentinel1/", RAW_DIR / "sentinel1")

# Azure Data Lake Storage Gen2 - credentials via DefaultAzureCredential (env vars, managed identity, az login)
# adls = get_connector("adls", account_url="https://myaccount.dfs.core.windows.net", filesystem="sar-data")
# adls.download_prefix("sentinel1/", RAW_DIR / "sentinel1")

# Google Cloud Storage - credentials via Application Default Credentials, or anonymous=True for public buckets
# gcs = get_connector("gcs", bucket="my-sar-bucket")
# gcs.download_prefix("sentinel1/", RAW_DIR / "sentinel1")

print("Connector examples above are real code, commented out because no cloud credentials are configured here.")

Connector examples above are real code, commented out because no cloud credentials are configured here.


## Real pull: sen1floods11 via the public-bucket HTTP connector

This is exactly how `datasets/raw/sen1floods11/` was populated (2,235 files, 1.7GB) - no SDK, no credentials, plain HTTPS against the GCS-compatible XML listing API. Re-running `list_objects()` here re-verifies the bucket is still reachable and shows the real object count/size without re-downloading (files are skipped if already present and the right size).

In [3]:
with logger.stage("verify_sen1floods11_source") as stage:
    http = get_connector("http", base_url="https://storage.googleapis.com/sen1floods11/")
    objects = http.list_objects("v1.1/data/flood_events/HandLabeled/S1Hand/")
    total_mb = sum(o.size_bytes for o in objects) / 1024 / 1024
    stage.metrics = {"remote_object_count": len(objects), "remote_size_mb": round(total_mb, 1)}

print(f"{len(objects)} S1Hand objects on the remote bucket, {total_mb:.1f} MB")

[01_ingest] -> verify_sen1floods11_source ...


[01_ingest] <- verify_sen1floods11_source [OK] 2.677s {'remote_object_count': 446, 'remote_size_mb': 695.7}
446 S1Hand objects on the remote bucket, 695.7 MB


## Catalog what's actually on local disk

`src/catalog/scanner.py` walks `datasets/raw/` and reports real per-source metadata (dates, polarizations, chip counts, event names) - not placeholders.

In [4]:
from src.catalog.scanner import scan_raw_datasets

with logger.stage("catalog_local_raw") as stage:
    records = scan_raw_datasets()
    stage.metrics = {"sources_found": len(records)}

for r in records:
    print(f"{r.id:45s} {r.source.value:14s} {r.status.value:10s}")

[01_ingest] -> catalog_local_raw ...
[01_ingest] <- catalog_local_raw [OK] 0.008s {'sources_found': 1}
sen1floods11                                  sen1floods11   raw       


In [5]:
logger.log_metrics({"local_sources": len(records)})
logger.finalize()

[01_ingest] run complete in 2.723s -> D:\project-raw-data\sphoorthq-geoverse\datasets\reports\runs\721e48e3-f36e-4602-ad60-54a6f9cde463.json


'D:\\project-raw-data\\sphoorthq-geoverse\\datasets\\reports\\runs\\721e48e3-f36e-4602-ad60-54a6f9cde463.json'